# 🏗️ Note — Spark Architecture: what happens when you *submit* a program

This is a **deep-dive note**, not a chapter: no new PySpark functions here, only the machinery underneath every chapter.
It expands the short architecture table at the top of [chapter7.ipynb](chapter7.ipynb) with the full story —
submit, JVM, executors, cores, tasks, success/failure, cluster managers, and the two deployment modes.

Source material: the *Spark Architecture* video of the PySpark playlist
([youtu.be/CYyUuInwgtA](https://youtu.be/CYyUuInwgtA?list=PL2IsFZBGM_IHCl9zhRVC1EXTomkEp_1zm)).

**Read in this order:**

| # | Section | Question it answers |
|---|---------|--------------------|
| 1 | Submitting a program | What does *"submit"* even mean? |
| 2 | JVM | What is a JVM, and why is an executor "a JVM"? |
| 3 | The 6-step lifecycle | Who talks to whom, in what order? |
| 4 | Executors vs cores | Is it 4 executors or 2? (the confusing bit) |
| 5 | "Copying the program" | What actually travels to the executors? |
| 6 | Success or failure | What counts as failure, and what does Spark do about it? |
| 7 | Resource manager | Is it responsible for the driver or the SparkSession? The 4 types. |
| 8 | Client vs cluster mode | Where does the driver *sit*? |
| 9 | spark-submit cheat sheet | Which flag maps to which part of the story? |
| 10 | Glossary + this course | Node / worker / executor / core / task, and where `local[*]` fits |

## 1. What does "submitting a program" mean?

A Spark program is just a file of code — say `daily_sales.py`. Running it is not like running a normal Python script,
because the code has to end up spread over many machines. **Submitting** is that hand-over step:

> You hand your program to Spark's launcher, and the launcher starts a **driver** process for it and asks a cluster for machines to work on.

The launcher is a command-line tool that ships with Spark: **`spark-submit`**.

```bash
spark-submit \
  --master yarn \                 # which cluster manager to talk to
  --deploy-mode client \          # where the driver should run (section 8)
  --num-executors 4 \             # I want 4 executors ...
  --executor-cores 2 \            # ... with 2 CPU cores each  (= 8 cores total)
  --executor-memory 4g \
  daily_sales.py  2026-08-13      # your program + its arguments
```

Nothing magic happened: `spark-submit` started **one JVM that runs your program** (the driver), and your program's
`SparkSession.builder.getOrCreate()` line then went shopping for executors with the numbers above.

### The three ways you will meet "submit"

| How you run Spark | What "submit" looks like |
|---|---|
| `spark-submit app.py` (production, scheduled jobs, Airflow) | Explicit. You type the command; the driver starts, runs, exits. |
| **A notebook** (this course, Jupyter, Databricks) | Invisible. The notebook kernel *is* the driver process, and it stays alive for hours. The "submit" happens the moment you run `SparkSession.builder...getOrCreate()`; the "exit" happens on `spark.stop()` or when you shut the kernel down. |
| A UI / API (Databricks job, Livy, `spark-submit` REST) | Someone else runs the equivalent of `spark-submit` for you. |

So in these notebooks you are always in the middle of an already-submitted application — which is exactly why
each chapter creates its own `SparkSession` at the top and (from chapter 9 on) calls `spark.stop()` at the bottom.

## 2. What is a JVM? (and why "an executor is a JVM")

**JVM = Java Virtual Machine** — a program whose job is to run other programs. Java and Scala code is compiled to
*bytecode* (not Windows/Linux machine code), and the JVM executes that bytecode while managing its own chunk of
memory (the *heap*) and cleaning it up (*garbage collection*). One JVM = **one operating-system process** with its
own memory limit.

Spark's engine is written in **Scala**, so everything Spark does actually happens inside JVMs:

- **Executor = one JVM process** on a worker machine. `--executor-memory 4g` is that JVM's heap size.
  Kill the JVM → the executor and everything cached in it is gone.
- **Driver = also a JVM** (plus a Python process, see below).
- Two executors on one node = two separate JVM processes on that node, isolated from each other.

### Where does Python fit in? (PySpark is a wrapper)

```text
 DRIVER  (your machine, or a cluster node)      EXECUTOR  (one JVM per box)
 ┌───────────────────────────────┐              ┌─────────────────────────────────┐
 │ your Python process           │              │ JVM                             │
 │   (this notebook)             │   tasks      │   reads the data, runs the plan │
 │        ↕  Py4J socket         │ ───────────▶ │   on its partitions             │
 │ JVM                           │              │        ↕  (ONLY for Python UDFs)│
 │   builds + optimises the plan │ ◀─────────── │   python worker process         │
 └───────────────────────────────┘   results    └─────────────────────────────────┘
```

- Your Python code only *describes* the work (`emp.filter(...).groupBy(...)`). It is sent through the **Py4J**
  bridge to the driver's JVM, which builds the query plan.
- The work itself runs in the executor JVMs — **no Python involved**.
- Exception: a **Python UDF** (`udf(lambda x: ...)`) can't run in the JVM, so each executor starts extra
  **python worker** processes and ships rows out to them and back. That serialisation is why built-in functions
  (`when`, `regexp_replace`, `to_date`, …) are much faster than Python UDFs.

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("Spark Architecture Notes").master("local[*]").getOrCreate())

In [ ]:
# Peek at the machinery of the session we just built
import os

sc = spark.sparkContext

print("Spark version          :", spark.version)
print("Application name       :", sc.appName)
print("Application id         :", sc.applicationId)          # the id the cluster manager tracks
print("master (cluster mgr)   :", sc.master)                 # local[*] = no real cluster
print("deploy mode            :", spark.conf.get("spark.submit.deployMode", "not set (local run)"))
print("default parallelism    :", sc.defaultParallelism)     # = usable cores => tasks that can run at once
print("CPU cores on this box  :", os.cpu_count())
print("Spark UI               :", sc.uiWebUrl)               # Executors / Environment tabs show all of this

# the JVM our Python process is talking to over Py4J
try:
    jvm = sc._jvm.java.lang.System
    print("JVM (java) version     :", jvm.getProperty("java.version"), "-", jvm.getProperty("java.vm.name"))
except Exception as e:
    print("JVM lookup failed      :", e)

## 3. The 6-step lifecycle of a submitted program

This is the whole story, in the order the video draws it. Assume we asked for **4 executors × 2 cores**.

```text
  (1) "I need 4 executors × 2 cores"        (2) "launch them on the nodes"
  ┌────────────┐  ────────────────────▶  ┌──────────────┐  ──────────────▶  ┌─────────────────────┐
  │   DRIVER   │                         │   CLUSTER    │                   │ Node-1  [exec][exec]│
  │ SparkSess. │  ◀────────────────────  │   MANAGER    │                   │ Node-2  [exec][exec]│
  └────────────┘  (3) "here they are:     └──────────────┘                   └─────────────────────┘
        │              host+port list"                                             ▲        │
        │  (4) connect, ship the code, then send tasks  ─────────────────────────────┘        │
        │  ◀───────────────────────  (5) results + status (success / failure)  ───────────────┘
        │
        └─ (6) "I'm finished, release them" ─▶ CLUSTER MANAGER kills the executors, frees the cores
```

| Step | What happens | Who starts it |
|------|--------------|---------------|
| **1** | The driver (holding the `SparkSession`) asks the cluster manager for resources: *N executors, C cores each, M memory each.* | Driver |
| **2** | The cluster manager finds free capacity on the worker nodes and **launches executor JVMs** there. | Cluster manager |
| **3** | The cluster manager reports back to the driver: which executors exist and how to reach them. Executors also register themselves directly with the driver. | Cluster manager / executors |
| **4** | The driver connects to the executors, **ships your code** to them (section 5), splits the job into **stages → tasks**, and hands one task per free core. | Driver |
| **5** | Each core processes **one partition** per task and reports back — the value/rows for an action, or a failure. | Executors |
| **6** | When the application ends (`spark.stop()`, script exits, or a failure), the driver tells the cluster manager to **release the resources**; the manager tears the executor JVMs down. | Driver → cluster manager |

Two things worth burning in:

- **Executors are per-application, not shared.** Another submitted program gets its own executor JVMs. That is why
  step 6 matters: unused resources held by an idle notebook are cores nobody else can use.
- **The driver never touches the data** (unless you call `collect()`, `show()`, `toPandas()` — those pull rows *to*
  the driver, which is how drivers run out of memory).

## 4. Executors vs cores — "is it 4 executors or 2?"

The video says both *"4 executors, 2 cores each = 8 cores"* and *"it created 2 executors per node"*, which sounds
contradictory. It isn't — the numbers count different things:

```text
                cluster
  ┌──────────────────────────────────────┐
  │  Node-1                Node-2        │
  │  ┌────────┐ ┌────────┐ ┌────────┐ ┌────────┐
  │  │ exec 1 │ │ exec 2 │ │ exec 3 │ │ exec 4 │   ← 4 executor JVMs in total
  │  │ ● ●    │ │ ● ●    │ │ ● ●    │ │ ● ●    │   ← 2 cores each
  │  └────────┘ └────────┘ └────────┘ └────────┘
  └──────────────────────────────────────┘
     2 executors per node  ×  2 nodes  =  4 executors  ×  2 cores  =  8 cores
```

| Word | Count in this example | Meaning |
|------|----------------------|---------|
| **Node / worker** | 2 | Physical (or virtual) machines offering CPU + RAM. |
| **Executor** | 4 total, 2 per node | JVM processes. `--num-executors 4` is the **total**, not per node — the manager decides how to spread them. |
| **Core** | 2 per executor, 8 total | Task slots. A "core" here is a thread the executor may use, not necessarily a dedicated physical CPU. |
| **Task** | 8 running at a time | One task = one partition. 8 cores → 8 partitions processed simultaneously. |

So with **20 partitions and 8 cores**, Spark runs 8 tasks, then 8, then 4 — three *waves*. This is the same
arithmetic as chapter 7's "20 partitions, 2 cores → 10 waves".

> ⚠️ `--num-executors` exists on YARN and Kubernetes. On **standalone** Spark you instead say
> `--total-executor-cores 8 --executor-cores 2` and Spark divides that into 4 executors. Same outcome, different dial.
> With **dynamic allocation** (`spark.dynamicAllocation.enabled=true`) you don't fix the number at all — Spark adds
> executors when tasks queue up and gives them back when they idle.

## 5. "It copies the Python program to every executor" — what actually travels?

Step 4 says the driver "copies the program to the executors". Precisely, three different things move:

| What moves | How | Note |
|---|---|---|
| **Your code** | Serialised with each task (closures/lambdas), plus any file you passed with `--py-files` / `--jars` / `--files`, which the cluster manager distributes to every executor before it starts. | This is why an executor can run *your* logic at all. |
| **The plan** | The driver's optimised physical plan, sliced into **stages** (cut at every shuffle) and then into **tasks** (one per partition). | Executors never plan; they only execute. |
| **Small lookup data** | `broadcast(df)` / broadcast variables — the driver sends one read-only copy to each executor. | The trick behind broadcast joins. |

**Your big data does *not* travel through the driver.** Each executor opens the source itself (S3/HDFS/local path)
and reads only its own partitions. The only data crossing the network between executors is **shuffle** data
(chapter 7), and the only data reaching the driver is what an action asks for.

## 6. "Based on the status — success or failure" — what does that mean?

Spark reports status at four nested levels:

```text
 application  (your whole submitted program)
   └── job     (one per action: show(), count(), write() ...)
        └── stage   (a chunk of the job between two shuffles)
             └── task   (one partition, on one core)
```

**A task fails** when the code running on that partition blows up. Real causes you will hit:

| Failure | Typical cause |
|---|---|
| Exception in your logic | dividing by zero, casting `"abc"` to int, key missing |
| Bad input data | corrupt record / wrong delimiter with `FAILFAST` (chapter 8) |
| `OutOfMemoryError` | a partition too big, a skewed key, `collect()` of a huge DataFrame |
| Executor lost | the JVM was killed (OOM killer, node reboot, spot/preemptible instance reclaimed) |
| I/O error | file deleted mid-read, permission denied, network timeout |

**What Spark does about it — it retries first.** A failed task is re-run on another executor up to
`spark.task.maxFailures` attempts (**default 4**). This is why a flaky node usually does not kill your job. If the
task still fails on the last attempt:

```text
 task fails 4×  →  its stage fails  →  its job fails  →  the application fails
```

The driver then aborts the job, prints the stack trace, marks the application **FAILED**, and (step 6) still asks
the cluster manager to release the executors. **Success** is the mirror image: every task of every stage finished,
the action returned its rows / the `write()` committed its files, and the application ends **SUCCEEDED**.

| Where you see the verdict | Client mode | Cluster mode |
|---|---|---|
| Stack trace | in your terminal / notebook output cell | in the cluster's driver log |
| Exit code | `spark-submit` returns non-zero on failure (`echo $?`) — this is what Airflow/cron checks | the launcher reports the final app status (on YARN it waits unless `spark.yarn.submit.waitAppCompletion=false`) |
| Web UI | `localhost:4040` while alive; the History Server afterwards | cluster manager UI (YARN RM / K8s / Spark Master) |

Practical habit: when a job fails, read the **first** failed task in the Spark UI's *Stages* tab, not the last error
in the console — the console usually shows the driver giving up, while the real cause is in the task.

## 7. The resource manager (a.k.a. cluster manager)

### "It is responsible for the driver program, not the SparkSession" — is that a mistake?

**No — it is a precision point, and a useful one.** A cluster manager deals with **operating-system processes**
(and the memory/cores they hold). The `SparkSession` is not a process: it is a **Python/Scala object living inside
the driver process**, your handle for building DataFrames.

```text
 ┌──────────── driver PROCESS  ← this is what the cluster manager sees, tracks and (in cluster
 │                                mode) launches; it appears in the manager's UI as one application
 │   ┌──── SparkSession OBJECT ← this is what your code sees: spark.read, spark.sql, spark.conf
 │   │        └── SparkContext  ← the piece that actually negotiates for executors
 └───┴──────────────────────────
```

Saying "the cluster manager talks to the driver" is correct. Saying "…to the SparkSession" is shorthand — harmless
in conversation, wrong when you are debugging, because in **cluster mode the manager launches the driver process
itself** (section 8), and it never knows or cares which objects live inside it.

### What the cluster manager actually does

- keeps a register of worker nodes and their **free cores + memory**
- receives resource requests from drivers and **queues** them when the cluster is full
- **launches and kills executor JVMs** on workers; restarts ones that die
- in **cluster** deploy mode: also launches (and can restart) the **driver**
- reports the application's final status and reclaims everything at the end

### The four types

| Cluster manager | What it is | Where you meet it |
|---|---|---|
| **Standalone** | Spark's own built-in manager: a `Master` process + `Worker` processes. Simple, Spark-only. | the [docker-images/](docker-images/) cluster in this repo (1 master, 2 workers) |
| **YARN** | Hadoop's resource manager — shares the cluster between Spark, Hive, MapReduce… | classic on-prem / EMR Hadoop clusters; still the most common in enterprises |
| **Mesos** | A general datacenter resource manager. **Deprecated in Spark 3.2 and removed in Spark 4.0** — know the name, don't learn it. | legacy systems only |
| **Kubernetes** | Containerised: the driver and every executor is a **pod**. | modern cloud deployments |

Two entries that belong on the same mental shelf:

- **`local[*]`** (these notebooks) is *not* a cluster manager. There is no manager, no network, no separate
  executor: driver and executor live in **one JVM**, and `*` means "use every core on this machine".
- **Databricks / EMR / Glue** hide the manager behind their own control plane, but the story is unchanged: a driver
  node plus executor JVMs on worker nodes.

## 8. Deployment modes: client vs cluster

Both modes run the *same* program with the *same* executors. The one difference:

> **Where does the driver process run — on the machine that submitted, or inside the cluster?**

### Client mode (`--deploy-mode client`, the default)

```text
  YOUR MACHINE (client)                    CLUSTER
  ─────────────────────                    ───────
  spark-submit
      │ starts
      ▼
  ┌─────────────┐   asks for resources   ┌─────────────────┐
  │   DRIVER    │ ─────────────────────▶ │ CLUSTER MANAGER │
  │ SparkSession│ ◀───────────────────── └────────┬────────┘
  └─────────────┘                                 │ launches
      ▲   │  tasks ──────────────▶  Node-1: [exec][exec]
      └───┴── results ◀──────────   Node-2: [exec][exec]

  ⚠️ the driver must stay alive for the whole run — close the laptop and the job dies
```

Everything described in sections 3–6 was client mode: **your machine keeps the driver**, so you see the logs live
and get results straight back. This is what a notebook does, and it is why an interrupted VPN or a closed lid kills
a running job.

### Cluster mode (`--deploy-mode cluster`)

```text
  YOUR MACHINE (client)                    CLUSTER
  ─────────────────────                    ───────
  spark-submit  ──── "here is my program" ──▶ ┌─────────────────┐
      │                                       │ CLUSTER MANAGER │
      └─ can now exit 🎉                      └────────┬────────┘
                                                       │ launches BOTH
                                     ┌─────────────────┴─────────────────┐
                                     ▼                                   ▼
                            Node-1                              Node-2
                            ┌──────────────┐                    ┌──────────────┐
                            │ DRIVER       │ ── tasks ────────▶ │ [exec][exec] │
                            │ SparkSession │ ◀── results ────── │              │
                            │ [exec]       │                    │              │
                            └──────────────┘                    └──────────────┘
```

The client uploads the program to the cluster manager and **its only job was to submit** — after that it may exit,
you may shut your laptop, and the application keeps running because the driver is a process *inside the cluster*.
When the run ends, the driver reports the final status to the cluster manager, which then tears down the driver and
the executors together.

> 🧹 **Clearing up the wording in the video:** "the driver sits inside an executor" is not literally true.
> In cluster mode the driver is its **own separate process** running **on a worker node** — a sibling of the
> executors, not inside one. It just *looks* like another box on a node in the drawing. Names it goes by:
> the **ApplicationMaster container** on YARN, a **driver pod** on Kubernetes, a driver process on a Worker in
> standalone mode. And note it consumes cluster resources too: `--driver-memory` / `--driver-cores` come out of the
> cluster in cluster mode, out of your own machine in client mode.

### Side by side

| | **Client mode** | **Cluster mode** |
|---|---|---|
| Driver runs on | the submitting machine (laptop, edge node, notebook kernel) | a node **inside** the cluster |
| After submit, the client… | must stay connected until the end | can exit immediately ("fire and forget") |
| Logs / stack traces | stream to your terminal or notebook cell | live in the cluster (`yarn logs -applicationId …`, `kubectl logs`, manager UI) |
| `collect()` / `show()` brings rows to | your machine's memory | the driver node's memory in the cluster |
| Driver ↔ executor network | across the office/VPN link — chatty and fragile | inside the datacenter — fast and stable |
| Good for | interactive work, notebooks, `spark-shell`, debugging | production and scheduled jobs (Airflow, cron, Databricks jobs) |
| Driver failure | your process dies; nothing restarts it | the manager can be configured to restart it |

And the third thing that is *not* a deploy mode: **local mode** (`master("local[*]")`, this course) — one JVM,
no cluster manager, no network. Perfect for learning, useless for scale.

In [ ]:
# Which mode am I in right now? Ask the session instead of guessing.
mode = spark.conf.get("spark.submit.deployMode", "unset")
master = spark.sparkContext.master

if master.startswith("local"):
    print(f"master={master!r} -> LOCAL mode: driver + executor in one JVM, no cluster manager, deployMode reported as {mode!r}")
else:
    print(f"master={master!r}, deploy mode={mode!r}")

# a few resource settings the driver asked for (or defaulted to)
for key in [
    "spark.master",
    "spark.app.name",
    "spark.driver.memory",
    "spark.executor.memory",
    "spark.executor.cores",
    "spark.executor.instances",
    "spark.dynamicAllocation.enabled",
    "spark.task.maxFailures",
    "spark.sql.shuffle.partitions",
]:
    try:
        # a key that was never set raises, unless Spark itself defines a default for it
        print(f"{key:35s} = {spark.conf.get(key)}")
    except Exception:
        print(f"{key:35s} = <not set: Spark's built-in default applies>")

## 9. `spark-submit` cheat sheet — flag → story

| Flag | Which part of the story it controls |
|---|---|
| `--master yarn` / `k8s://…` / `spark://host:7077` / `local[*]` | **which cluster manager** the driver negotiates with (section 7) |
| `--deploy-mode client\|cluster` | **where the driver runs** (section 8) |
| `--num-executors 4` | how many executor JVMs to ask for in step 1 (YARN/K8s) |
| `--executor-cores 2` | task slots per executor → parallelism (section 4) |
| `--executor-memory 4g` | heap size of each executor JVM (section 2) |
| `--driver-memory 2g`, `--driver-cores 1` | the driver's own JVM — raise it before `collect()`ing a lot |
| `--total-executor-cores 8` | the standalone-mode way to size the app |
| `--py-files utils.zip`, `--jars a.jar`, `--files config.yaml` | extra things shipped to every executor (section 5) |
| `--conf spark.dynamicAllocation.enabled=true` | let Spark grow/shrink the executor count on demand |
| `--name daily_sales` | the application name in the manager UI and History Server |
| `app.py arg1 arg2` | your program and its arguments — always last |

Rule of thumb for sizing: **many small executors** parallelise better and lose less when one dies; **few fat
executors** cache more and shuffle less. The usual compromise is 2–5 cores per executor, never 1 giant executor
per node.

## 10. Glossary + where this course sits

| Word | Precisely | Common sloppy use |
|---|---|---|
| **Node** | one machine in the cluster | used interchangeably with "worker" |
| **Worker** | the manager's agent process on a node, which hosts executors | called "the node" |
| **Executor** | one JVM process doing data work for **one** application | called "a node" or "a core" |
| **Core** | one task slot inside an executor | confused with a physical CPU |
| **Slot** | exactly the same thing as a core, in scheduling talk | — |
| **Task** | work for **one partition** on one core | confused with "job" |
| **Stage** | a group of tasks with no shuffle in between | confused with "job" |
| **Job** | everything triggered by **one action** | used for the whole application |
| **Application** | one submitted program = one driver + its executors | — |
| **Driver** | the process running your code and the plan | confused with the master |
| **Master / resource manager / cluster manager** | the resource negotiator — three names, one role | confused with the driver |

### The three setups in this repo

| Setup | Cluster manager | Driver | Executors |
|---|---|---|---|
| `master("local[*]")` — chapters 1-10 | none (local mode) | your Python + JVM process | none: the same JVM does the work, `*` cores |
| [docker-images/](docker-images/) | standalone (1 master + 2 workers) | the Jupyter container (client mode) | JVMs inside the worker containers |
| Databricks Community (chapter 6 tip) | Databricks' own | the cluster's driver node | single-node cluster: the driver's own cores |

### Watch it happen

Open the Spark UI (`localhost:4040`, see [jup_Note_url.txt](jup_Note_url.txt)) while a job runs:

- **Executors** tab → every executor JVM, its cores, its memory, tasks completed/failed — section 3 and 4, live.
- **Environment** tab → every `--conf` and flag the driver was launched with — section 9.
- **Jobs / Stages** tabs → the job → stage → task tree, with the wave pattern of section 4 and any retries from section 6.

In [ ]:
# Free the resources — step 6 of the lifecycle, done by hand.
# The driver tells the cluster manager it is finished; executors get torn down.
spark.stop()
print("SparkSession stopped -> executors released, Spark UI on :4040 is gone")